# Final project alignment and model comparison
Read the team guide for architecture and limitations. This notebook uses saved evidence and weights; it does not retrain models.

In [1]:
from pathlib import Path
import sys,json
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'peptide_lab').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from peptide_lab.config import RESULTS
from peptide_lab.pipeline import ScreeningPipeline
display(pd.read_csv(RESULTS/'alignment/final_model_comparison.csv'))

,task,model,validation_auc,accuracy,precision,recall,f1,roc_auc,mcc
0,activity,Compact ensemble,0.994395,0.961924,0.945892,0.920796,0.933175,0.991557,0.906720
1,activity,ESM-2 + CNN,0.992945,0.951786,0.883028,0.960203,0.920000,0.990339,0.887174
2,toxicity,Compact cnn,0.968533,0.922414,0.937500,0.892193,0.914286,0.971802,0.844362
3,toxicity,ESM-2 + CNN,0.957356,0.898276,0.883212,0.899628,0.891344,0.966877,0.795855


In [2]:
print(json.dumps(json.loads((RESULTS/'alignment/property_thresholds.json').read_text()),indent=2))
print(json.dumps(json.loads((RESULTS/'generation_audit.json').read_text()),indent=2))

{
  "compact": {
    "activity": {
      "threshold": 0.49,
      "rule": "Maximize validation F1 over 0.01..0.99; ties use higher cutoff.",
      "validation_rows": 2668,
      "validation_sha256": "f268e0423c55fbca564bc32e797e3f94a604375f9a47b304c744ad7aab5bb2c6",
      "recall": 0.9442282749675746,
      "precision": 0.9516339869281045,
      "f1": 0.9479166666666666,
      "protocol": "Follow-up validation-only screening rule; benchmark classification metrics retain threshold 0.5. Not a safety guarantee."
    },
    "toxicity": {
      "threshold": 0.08,
      "rule": "Highest grid cutoff with validation toxic-class recall >= 0.95; fallback zero.",
      "validation_rows": 657,
      "validation_sha256": "5315ec5cf86727c9d2a85e3b04d2155bd27ea1bc7ff1d666a20105bf75a5b7d4",
      "recall": 0.9519519519519519,
      "precision": 0.8637602179836512,
      "f1": 0.9057142857142857,
      "protocol": "Follow-up validation-only screening rule; benchmark classification metrics retain thresh

## Both engines can screen the same existing sequences
Labels belong to separate tasks; scores are not calibrated safety probabilities.

In [3]:
sequences=['GILSGILGAGKSLVCGLSGLC','KWKLFKKIGAVLKVL']
for engine in ['compact','esm2']:
    pipe=ScreeningPipeline(engine,ranking_rule='weighted_difference')
    display(pipe.predict(sequences)[['rank','sequence','prediction_engine','activity_score','toxicity_score','priority_score','passes_property_screen']])

,rank,sequence,prediction_engine,activity_score,toxicity_score,priority_score,passes_property_screen
0,1,GILSGILGAGKSLVCGLSGLC,compact,0.999959,0.020718,98.962037,True
1,2,KWKLFKKIGAVLKVL,compact,0.999997,0.081220,95.938815,False


,rank,sequence,prediction_engine,activity_score,toxicity_score,priority_score,passes_property_screen
0,1,GILSGILGAGKSLVCGLSGLC,esm2,1.0,0.003499,99.825027,True
1,2,KWKLFKKIGAVLKVL,esm2,1.0,0.972170,51.391518,False


In [4]:
display(pd.DataFrame(json.loads((RESULTS/'alignment/similarity_summary.json').read_text())['tasks']))
display(pd.read_csv(RESULTS/'alignment/amino_acid_composition.csv'))

,task,test_rows,training_rows,test_with_training_similarity_ge_0_8,fraction
0,activity,8877,10671,943,0.106230
1,toxicity,580,2627,118,0.203448
2,vae,651,5208,421,0.646697


,amino_acid,training,generated
0,A,0.077524,0.074624
1,C,0.038307,0.024183
2,D,0.015608,0.013077
3,E,0.015656,0.014219
4,F,0.051303,0.057187
5,G,0.088625,0.084276
6,H,0.022307,0.020965
7,I,0.066212,0.065490
8,K,0.148435,0.182667
9,L,0.114281,0.132641
